# VLM Eval — Interactive Viewer

Browse generated records from `outputs/vlm_eval_hf/`.
Filter by model / decoding / split / task / error, zoom the image, and step through results.


In [1]:
import io
import json
import logging
from collections import defaultdict
from pathlib import Path

import ipywidgets as widgets
from datasets import load_dataset
from IPython.display import display
from PIL import Image as _PILImage

logging.getLogger("transformers.generation.utils").setLevel(logging.ERROR)

# VS Code Jupyter runs with CWD = workspace root (FineSightBench/), same as the inference notebook.
OUTPUT_DIR = Path("outputs/vlm_eval_hf")

# ── Load all JSONL records ────────────────────────────────────────────────────
_records: list[dict] = []
for _jsonl in sorted(OUTPUT_DIR.glob("*.jsonl")):
    with _jsonl.open(encoding="utf-8") as _f:
        for _line in _f:
            _line = _line.strip()
            if not _line:
                continue
            try:
                _records.append(json.loads(_line))
            except json.JSONDecodeError:
                continue
print(f"Loaded {len(_records)} records from {OUTPUT_DIR}")

by_model: dict[tuple, int] = defaultdict(int)
for r in _records:
    by_model[(r.get("model"), r.get("decoding", "?"))] += 1
for (m, d), n in sorted(by_model.items()):
    print(f"  {m:40s}  {d:12s}  {n}")


Loaded 393 records from outputs/vlm_eval_hf
  GLM-4.6V-Flash                            greedy        24
  GLM-4.6V-Flash                            sample_t01    24
  GLM-4.6V-Flash                            sample_t10    21
  InternVL3_5-1B-Flash                      greedy        24
  InternVL3_5-1B-Flash                      sample_t01    24
  InternVL3_5-1B-Flash                      sample_t10    24
  Qwen3-VL-2B-Instruct                      greedy        60
  Qwen3-VL-2B-Instruct                      sample_t01    60
  Qwen3-VL-2B-Instruct                      sample_t10    60
  gemma-4-E2B-it                            greedy        24
  gemma-4-E2B-it                            sample_t01    24
  gemma-4-E2B-it                            sample_t10    24


In [2]:
# ── Cache HF datasets for image lookup ───────────────────────────────────────
_ds_cache: dict[str, object] = {}

def _get_image(rec: dict) -> "_PILImage.Image | None":
    key = rec.get("dataset_id")
    if key is None:
        return None
    if key not in _ds_cache:
        _ds_cache[key] = load_dataset(key)
    split = rec.get("split")
    idx = rec.get("row_index")
    try:
        return _ds_cache[key][split][int(idx)]["image"]
    except Exception:
        return None


def _unique(values):
    seen: list = []
    for v in values:
        if v not in seen:
            seen.append(v)
    return seen


_ALL = "(all)"
_models_opts    = [_ALL] + _unique(r.get("model")     for r in _records)
_decodings_opts = [_ALL] + _unique(r.get("decoding")  for r in _records)
_splits_opts    = [_ALL] + _unique(r.get("split")     for r in _records)
_tasks_opts     = [_ALL] + sorted({r.get("task_type") for r in _records if r.get("task_type")})
_error_opts     = [_ALL, "only errors", "no errors"]

_model_w  = widgets.Dropdown(options=_models_opts,    value=_ALL, description="model")
_dec_w    = widgets.Dropdown(options=_decodings_opts,  value=_ALL, description="decoding")
_split_w  = widgets.Dropdown(options=_splits_opts,    value=_ALL, description="split")
_task_w   = widgets.Dropdown(options=_tasks_opts,     value=_ALL, description="task")
_err_w    = widgets.Dropdown(options=_error_opts,     value=_ALL, description="error")
_prev_btn = widgets.Button(description="◀ Prev", layout=widgets.Layout(width="90px"))
_next_btn = widgets.Button(description="Next ▶", layout=widgets.Layout(width="90px"))
_pos_lbl  = widgets.Label(value="0 / 0")

_zoom_w = widgets.IntSlider(
    value=320, min=100, max=1200, step=40,
    description="zoom px",
    continuous_update=True,
    layout=widgets.Layout(width="300px"),
    style={"description_width": "60px"},
)

_img_w  = widgets.Image(format="png", layout=widgets.Layout(
    width=f"{_zoom_w.value}px",
    max_width="100%",
    border="1px solid #ddd",
))
_meta_w = widgets.HTML()

_state = {"filtered": [], "pos": 0}


def _apply_filters() -> list[dict]:
    rows = list(_records)
    if _model_w.value != _ALL:
        rows = [r for r in rows if r.get("model") == _model_w.value]
    if _dec_w.value != _ALL:
        rows = [r for r in rows if r.get("decoding") == _dec_w.value]
    if _split_w.value != _ALL:
        rows = [r for r in rows if r.get("split") == _split_w.value]
    if _task_w.value != _ALL:
        rows = [r for r in rows if r.get("task_type") == _task_w.value]
    if _err_w.value == "only errors":
        rows = [r for r in rows if r.get("error")]
    elif _err_w.value == "no errors":
        rows = [r for r in rows if not r.get("error")]
    return rows


def _render():
    rows = _state["filtered"]
    total = len(rows)
    if total == 0:
        _pos_lbl.value = "0 / 0"
        _img_w.value = b""
        _meta_w.value = "<i>No records match the current filters.</i>"
        return
    _state["pos"] = max(0, min(_state["pos"], total - 1))
    pos = _state["pos"]
    rec = rows[pos]
    _pos_lbl.value = f"{pos + 1} / {total}"

    img = _get_image(rec)
    if img is not None:
        buf = io.BytesIO()
        img.convert("RGB").save(buf, format="PNG")
        _img_w.value = buf.getvalue()
    else:
        _img_w.value = b""

    def _esc(v):
        if v is None:
            return "<i>None</i>"
        s = str(v)
        return (s.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
                 .replace("\n", "<br>"))

    err_html = (f"<div style='color:#b00'><b>error:</b> {_esc(rec.get('error'))}</div>"
                if rec.get("error") else "")

    _meta_w.value = f"""
    <div style='font-family: system-ui, sans-serif; font-size: 13px; line-height:1.5'>
      <div><b>model</b>: {_esc(rec.get('model'))} &nbsp;
           <b>decoding</b>: {_esc(rec.get('decoding'))} (sample={_esc(rec.get('do_sample'))}, T={_esc(rec.get('temperature'))}) &nbsp;
           <b>split</b>: {_esc(rec.get('split'))} &nbsp;
           <b>task</b>: {_esc(rec.get('task_type'))} &nbsp;
           <b>difficulty</b>: {_esc(rec.get('difficulty'))}</div>
      <div><b>image_id</b>: {_esc(rec.get('image_id'))} &nbsp;
           <b>row_index</b>: {_esc(rec.get('row_index'))} &nbsp;
           <b>latency</b>: {_esc(rec.get('latency_s'))}s</div>
      <hr>
      <div><b>Question</b><br>{_esc(rec.get('question'))}</div>
      <div style='margin-top:8px;padding:8px;background:#eef7ee;border-radius:4px'>
        <b>Reference answer</b><br>{_esc(rec.get('reference_answer'))}
      </div>
      <div style='margin-top:8px;padding:8px;background:#eef2fb;border-radius:4px'>
        <b>Generated ({_esc(rec.get('model'))} / {_esc(rec.get('decoding'))})</b><br>{_esc(rec.get('generated_text'))}
      </div>
      {err_html}
    </div>
    """


def _on_filter_change(_change):
    _state["filtered"] = _apply_filters()
    _state["pos"] = 0
    _render()


def _on_prev(_b):
    if _state["filtered"]:
        _state["pos"] = (_state["pos"] - 1) % len(_state["filtered"])
        _render()


def _on_next(_b):
    if _state["filtered"]:
        _state["pos"] = (_state["pos"] + 1) % len(_state["filtered"])
        _render()


def _on_zoom_change(change):
    _img_w.layout.width = f"{change['new']}px"


for _w in (_model_w, _dec_w, _split_w, _task_w, _err_w):
    _w.observe(_on_filter_change, names="value")
_prev_btn.on_click(_on_prev)
_next_btn.on_click(_on_next)
_zoom_w.observe(_on_zoom_change, names="value")

_state["filtered"] = _apply_filters()
_render()

_controls = widgets.HBox([_model_w, _dec_w, _split_w, _task_w, _err_w])
_nav      = widgets.HBox([_prev_btn, _pos_lbl, _next_btn, _zoom_w])
_body     = widgets.HBox([_img_w, _meta_w],
                         layout=widgets.Layout(align_items="flex-start"))
display(widgets.VBox([_controls, _nav, _body]))
